# 04 · MLP (feed-forward)

**Peça nova:** `FeedForward`. A atenção é **comunicação**: cada posição junta informação das anteriores. Mas, depois de juntar, a posição quase não "processa" o que recebeu: a saída da atenção vai direto para a camada de saída.

A MLP é **computação**: aplicada a cada posição separadamente (as posições não conversam aqui), ela expande o vetor para `mult × d_model` dimensões, aplica uma não linearidade e volta para `d_model`. Num Transformer, a maior parte dos parâmetros mora aqui.

Daqui em diante, `MultiHeadAttention` vem pronta de `lab.model`. Diário: `F1-MLP`.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn

from lab.data import shakespeare
from lab.model import LanguageModel, MultiHeadAttention, count_params
from lab.plot import compare, table
from lab.train import pick_device, train

DEVICE = pick_device()
ds = shakespeare()
tok, V = ds.tok, ds.tok.vocab_size
BLOCK, D, STEPS, LR = 32, 64, 3000, 1e-3
print(ds, "| dispositivo:", DEVICE)

## A peça: `FeedForward`

In [ ]:
ACTIVATIONS = {"relu": nn.ReLU, "gelu": nn.GELU, "identity": nn.Identity}


class FeedForward(nn.Module):
    def __init__(self, d_model, mult=4, activation="relu"):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, mult * d_model),   # expande
            ACTIVATIONS[activation](),            # não linearidade
            nn.Linear(mult * d_model, d_model),   # volta
        )

    def forward(self, x):
        return self.net(x)  # (B, T, d_model) -> (B, T, d_model), cada posição independente

### Teste: a MLP não mistura posições
Mudar a posição 3 da entrada só pode mudar a posição 3 da saída.

In [ ]:
torch.manual_seed(0)
ff = FeedForward(D)
x = torch.randn(1, 8, D)
x2 = x.clone()
x2[:, 3] = torch.randn(D)
changed = (ff(x) - ff(x2)).abs().sum(-1)[0] > 0
print("posições que mudaram:", changed.nonzero().flatten().tolist())
print(f"parâmetros da MLP: {count_params(ff):,}  |  da atenção com 4 cabeças: {count_params(MultiHeadAttention(D, 4, BLOCK)):,}")

## O modelo: atenção, depois MLP
A opção `attention=False` serve ao experimento 3.

In [ ]:
class AttnMLPLM(LanguageModel):
    def __init__(self, vocab_size, block_size=32, d_model=64, n_head=4, mult=4, activation="relu", attention=True, mlp=True):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.sa = MultiHeadAttention(d_model, n_head, block_size) if attention else nn.Identity()
        self.ffwd = FeedForward(d_model, mult, activation) if mlp else nn.Identity()
        self.lm_head = nn.Linear(d_model, vocab_size)

    def logits(self, idx):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.sa(x)     # comunicação
        x = self.ffwd(x)   # computação
        return self.lm_head(x)

## Treino de referência

In [ ]:
torch.manual_seed(1337)
model = AttnMLPLM(V, BLOCK, D)
run = train(model, ds, name="04-mlp", steps=STEPS, lr=LR, device=DEVICE)
print(run.sample)

In [ ]:
compare(["01-bigram", "03-cabecas", "04-mlp"], title="Atenção sem e com MLP")
table(["01-bigram", "03-cabecas", "04-mlp"])

---
## Experimentos

### 1. Largura da MLP (`mult`)
**Minha previsão:** _(com mult 1, 2, 4 e 8: quanto cresce o número de parâmetros e quanto melhora a loss? aparece overfitting?)_

In [ ]:
for mult in [1, 2, 4, 8]:
    torch.manual_seed(1337)
    m = AttnMLPLM(V, BLOCK, D, mult=mult)
    print(f"mult {mult}: {count_params(m):,} parâmetros")
    train(m, ds, name=f"04-mult{mult}", steps=STEPS, lr=LR, device=DEVICE, verbose=False, sample_tokens=0)
compare(prefix="04-mult", title="Largura da MLP")
table(prefix="04-mult")

### 2. A não linearidade importa?
Com `identity`, as duas camadas lineares da MLP viram uma só multiplicação de matrizes (W₂·W₁ é uma matriz).

**Minha previsão:** _(identity fica igual ao modelo sem MLP? ReLU vs. GELU faz diferença aqui?)_

In [ ]:
for act in ["relu", "gelu", "identity"]:
    torch.manual_seed(1337)
    train(AttnMLPLM(V, BLOCK, D, activation=act), ds, name=f"04-act-{act}", steps=STEPS, lr=LR, device=DEVICE, verbose=False, sample_tokens=0)
compare(["03-cabecas"] + [f"04-act-{a}" for a in ["relu", "gelu", "identity"]], title="Ativação da MLP")
table(["03-cabecas"] + [f"04-act-{a}" for a in ["relu", "gelu", "identity"]])

### 3. MLP sem atenção
Sem atenção, cada posição só enxerga o próprio caractere (e a própria posição). É um bigrama com mais parâmetros.

**Minha previsão:** _(a loss fica perto da do bigrama ou da do modelo com atenção?)_

In [ ]:
torch.manual_seed(1337)
train(AttnMLPLM(V, BLOCK, D, attention=False), ds, name="04-so-mlp", steps=STEPS, lr=LR, device=DEVICE, verbose=False, sample_tokens=0)
compare(["01-bigram", "04-so-mlp", "04-mlp"], title="Só MLP vs. bigrama vs. atenção + MLP")
table(["01-bigram", "04-so-mlp", "04-mlp"])

---
## Para o diário (`F1-MLP-01`)

- Quanto a MLP acrescentou? Compare o ganho com o que a atenção trouxe no notebook 02.
- O resultado do `identity` confirmou que a não linearidade é o que dá poder à MLP?
- Por que "só MLP" fica perto do bigrama? O que isso diz sobre o papel de cada peça?

**Próximo:** [05 · residual](05_residual.ipynb), onde empilhamos vários blocos.